In [ ]:
import os
from dotenv import load_dotenv
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI
from langgraph.prebuilt import create_react_agent

In [ ]:
load_dotenv()

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [ ]:
from typing import Sequence, Union, Dict, Any, Callable, Optional
from langchain_core.tools import BaseTool
from langchain_core.utils.function_calling import format_tool_to_openai_tool
from langchain.schema.runnable import Runnable
from langchain.schema import BaseMessage

class ChatOpenRouter(ChatOpenAI):
    def __init__(self, **kwargs):
        super().__init__(
            model_name=kwargs.get("model_name", "openai/gpt-4o"),
            openai_api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1",
            **kwargs,
        )
    
    def bind_tools(
        self,
        tools: Sequence[Union[Dict[str, Any], Callable, BaseTool]],
        *,
        tool_choice: Optional[
            Union[dict, str, bool]
        ] = None,
        **kwargs: Any,
    ):
        """
        Привязка инструментов к модели в формате OpenAI tool-calling API.
        """
        # Конвертация инструментов в формат, который ожидает модель
        formatted_tools = [format_tool_to_openai_tool(tool) for tool in tools]
        
        if tool_choice is not None:
            if isinstance(tool_choice, str) and tool_choice not in ("auto", "any", "none"):
                tool_choice = {"type": "function", "function": {"name": tool_choice}}
            if isinstance(tool_choice, bool):
                if len(tools) != 1:
                    raise ValueError("tool_choice=True возможен только с одним инструментом")
                tool_name = formatted_tools[0]["function"]["name"]
                tool_choice = {"type": "function", "function": {"name": tool_name}}
            kwargs["tool_choice"] = tool_choice
        
        # Возвращаем runnable объект с привязкой инструментов
        return super().bind(tools=formatted_tools, **kwargs)


In [ ]:
# Здесь определяется калькулятор и генератор кода как функции/инструменты (пример через функцию)
def calculator_tool(query: str) -> str:
    """
    Инструмент калькулятора: вычисляет арифметическое выражение,
    переданное в query, безопасно с обработкой ошибок.
    """
    try:
        return str(eval(query))
    except Exception as e:
        return f"Ошибка: {e}"
    
tools = [calculator_tool]

llm = ChatOpenRouter()

agent = create_react_agent(llm, tools)

response = await agent.ainvoke({
    "messages": [
        {"role": "system", "content": "using calculator_tool for calculate"},
        {"role": "user", "content": "100*5/2"}
    ]
})

print(response['messages'][-1].content)

TypeError: create_react_agent() got unexpected keyword arguments: {'verbose': True}

In [44]:
str(eval("100*5/2"))

'250.0'

In [ ]:
import re
from langchain.agents import Tool, initialize_agent
from langchain.agents.agent_types import AgentType
# from langchain_ollama import OllamaLLM
import num2words

# 1. Настройка локальной модели Ollama через langchain-ollama
# llm = OllamaLLM(
#     model="llama3.1",
#     # base_url="http://localhost:11434",
#     timeout=120
# )

def calculator_fn(text: str) -> str:
    """
    Evaluate a mathematical expression written in a natural language.

    Supported formats:
      - "X в степени Y"
      - "X^Y"
      - Python expressions with '**' operator

    Return the result of the expression as a string.
    If the expression is invalid, return an error message.
    """
    txt = text.strip().lower()
    m = re.search(r"(\d+)\s*в степени\s*(\d+)", txt)
    if m:
        base, exp = map(int, m.groups())
        return str(pow(base, exp))
    m = re.search(r"(\d+)\s*\^\s*(\d+)", txt)
    if m:
        base, exp = map(int, m.groups())
        return str(pow(base, exp))
    try:
        expr = txt.replace('^', '**')
        return str(eval(expr))
    except Exception as e:
        return f"Ошибка вычисления: {e}"

# 2. Функция перевода числа в английское слово
def translate_number_to_english(number):
    return num2words.num2words(number)

# 3. Инструменты LangChain
calculator = Tool(
    name="calculator",
    func=calculator_fn,
    description="Вычисляет арифметические выражения: поддерживает 'X в степени Y', 'X^Y' и Python-выражения."
)
translator = Tool(
    name="translator",
    func=translate_number_to_english,
    description="Переводит число в английское слово и напрямую возвращает результат в качестве окончательного ответа агента.",
    return_direct=True  # Агент завершает работу сразу после вызова переводчика
)

tools = [calculator, translator]

# 4. Инициализация агента ReAct
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=4,
    early_stopping_method="force"
    )

print(agent.run("Посчитай, сколько будет 6 в степени 13, а результат переведи на английский"))




> Entering new AgentExecutor chain...
To answer this question, I need to add the numbers 6 and 13, and then translate the sum into English words.

Action: calculator
Action Input: 6 + 13
Observation: 19
Thought:I have calculated the sum of 6 and 13, which is 19. Now, I need to translate the number 19 into English words.

Action: translator
Action Input: 19
Observation: nineteen


> Finished chain.
nineteen
